# Lab 03 — マスターと結合の検証（読み取り専用）

ゾーンマスターのキー品質と、乗車・降車の 2 つの役割での結合結果を確認します。データは変更しません。

In [ ]:
import json
from datetime import datetime, timezone

def save_result(name, payload):
    """Persist a run summary that scripts/lab.py reads back from OneLake."""
    payload = {"notebook": name, "finishedAtUtc": datetime.now(timezone.utc).isoformat(timespec="seconds"), **payload}
    notebookutils.fs.put(f"Files/lab_results/{name}.json", json.dumps(payload, ensure_ascii=False, default=str), True)
    print(json.dumps(payload, ensure_ascii=False, indent=2, default=str))

checks = []

def check(name, expected, actual, tolerance=None):
    if tolerance is None:
        passed = expected == actual
    else:
        passed = actual is not None and abs(float(expected) - float(actual)) <= tolerance
    checks.append({"check": name, "expected": expected, "actual": actual, "passed": bool(passed)})

In [ ]:
from pyspark.sql import functions as F

zones = spark.table("taxi_zone_lookup")
trips = spark.table("green_tripdata_2026")
check("zone rows", 265, zones.count())
key_quality = zones.agg(F.countDistinct("LocationID").alias("distinct"),
                        F.sum(F.col("LocationID").isNull().cast("int")).alias("nulls")).first()
check("zone LocationID distinct", 265, key_quality.distinct)
check("zone LocationID nulls", 0, key_quality.nulls)
EXPECTED_ZONES = {1: ("EWR", "Newark Airport", "EWR"), 2: ("Queens", "Jamaica Bay", "Boro Zone"),
                  3: ("Bronx", "Allerton/Pelham Gardens", "Boro Zone"), 4: ("Manhattan", "Alphabet City", "Yellow Zone"),
                  5: ("Staten Island", "Arden Heights", "Boro Zone")}
fixed = {r.LocationID: (r.Borough, r.Zone, r.service_zone)
         for r in zones.filter(F.col("LocationID").isin(*EXPECTED_ZONES)).orderBy("LocationID").collect()}
check("fixed zones 1-5", {str(k): list(v) for k, v in EXPECTED_ZONES.items()}, {str(k): list(v) for k, v in fixed.items()})
for table, rows in {"vendor_lookup": 2, "ratecode_lookup": 6, "payment_type_lookup": 6, "trip_type_lookup": 2}.items():
    check(f"{table} rows", rows, spark.table(table).count())

## 乗車地・降車地の 2 つの役割で結合する

In [ ]:
pickup = zones.select(F.col("LocationID").alias("PULocationID"), F.col("Borough").alias("pickup_borough"),
                      F.col("Zone").alias("pickup_zone"))
dropoff = zones.select(F.col("LocationID").alias("DOLocationID"), F.col("Borough").alias("dropoff_borough"),
                       F.col("Zone").alias("dropoff_zone"))
joined = trips.join(pickup, "PULocationID", "inner").join(dropoff, "DOLocationID", "inner")
check("joined rows (no loss, no duplication)", 37948, joined.count())

boroughs = {r.pickup_borough: r["count"] for r in joined.groupBy("pickup_borough").count().collect()}
MAIN = {"Queens": 9957, "Manhattan": 9937, "Brooklyn": 8702, "Bronx": 6151, "Staten Island": 2800, "EWR": 135}
check("pickup borough trips", MAIN, {k: boroughs.get(k) for k in MAIN})
check("pickup borough other (Unknown / N/A)", 266, sum(v for k, v in boroughs.items() if k not in MAIN))

mb = joined.filter((F.col("pickup_borough") == "Manhattan") & (F.col("dropoff_borough") == "Brooklyn"))
summary = mb.agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("amount")).first()
check("Manhattan -> Brooklyn trips", 2340, summary.trips)
check("Manhattan -> Brooklyn total_amount", 42944.85, float(summary.amount), 0.005)

fixed_trips = {r.trip_id: [r.pickup_zone, r.pickup_borough, r.dropoff_zone, r.dropoff_borough]
               for r in joined.filter(F.col("trip_id").isin(1, 4)).collect()}
check("fixed trip zones", {"1": ["Washington Heights North", "Manhattan", "Gravesend", "Brooklyn"],
                           "4": ["Saint George/New Brighton", "Staten Island", "Flatbush/Ditmas Park", "Brooklyn"]},
      {str(k): v for k, v in sorted(fixed_trips.items())})

failed = [c for c in checks if not c["passed"]]
save_result("nb_03_validate_dimensions", {"passed": not failed, "failed": len(failed), "checks": checks})